---
title: "广度优先搜索 (BFS)——拓扑排序"
format:
  html:
    embed-resources: true
    toc: true
    theme: cosmo
    code-copy: true
  pdf:
    pdf-engine: xelatex
    documentclass: article
---

In [1]:
#| code-fold: true
from collections import deque
from typing import List

## 📝 题目 210：课程表 II

::: {.callout-caution}
### 题目要求
**描述**：
现在你总共有 `numCourses` 门课需要选，记为 `0` 到 `numCourses - 1`。
给你一个数组 `prerequisites` ，其中 `prerequisites[i] = [ai, bi]` ，表示在选修课程 `ai` 前 **必须** 先选修 `bi` 。
- 例如 `[1, 0]` 表示：想上课程 1，必须先完成课程 0。

**返回目标**：

- 返回你为了学完所有课程所安排的 **学习顺序**。
- 如果有多个正确的顺序，只要返回 **任意一个** 即可。
- 如果由于“环”的存在（例如 A 依赖 B，B 依赖 A）导致无法完成所有课程，返回 **空数组 `[]`**。

**输入输出示例**：

* **示例 1**：
    * **输入**：`numCourses = 2, prerequisites = [[1, 0]]`
    * **输出**：`[0, 1]`
    * **说明**：总共有 2 门课程。要学习课程 1，你需要先完成课程 0。因此，正确的课程顺序为 [0, 1] 。

* **示例 2**：
    * **输入**：`numCourses = 4, prerequisites = [[1,0],[2,0],[3,1],[3,2]]`
    * **输出**：`[0,1,2,3]` 或 `[0,2,1,3]`
    * **说明**：总共有 4 门课程。要学习课程 3，你应该先完成课程 1 和 2。并且课程 1 和 2 都应该排在课程 0 之后。因此，一个正确的课程顺序是 [0,1,2,3] 。另一个正确的只需满足顺序要求即可。

* **示例 3**：
    * **输入**：`numCourses = 1, prerequisites = []`
    * **输出**：`[0]`
:::

In [2]:
class Solution210:
    def findOrder(self, n:int, pre:List[List[int]]) -> List[int]:
        indegree = [0] * n  # 准备入度表
        adj = [[] for _ in range(n)]  # 准备邻接表
        for ai,bi in pre:
            indegree[ai] += 1  # 后续课的入度 +1
            adj[bi].append(ai)  # 先修课指向后续课
        queue = deque()
        for i in range(n):  # 寻找起点
            if indegree[i] == 0:
                queue.append(i)
        res = []
        while queue:
            curr = queue.popleft()
            res.append(curr)
            for nxt in adj[curr]:
                indegree[nxt] -= 1  # 更新后续课的入度
                if indegree[nxt] == 0:
                    queue.append(nxt)
        return res if len(res) == n else []

In [7]:
#| code-fold: true
def test_210(func):
    test_cases = [
        {"n": 2, "pre": [[1,0]], "expected_one_of": [[0,1]], "desc": "基础依赖"},
        {"n": 4, "pre": [[1,0],[2,0],[3,1],[3,2]], "expected_one_of": [[0,1,2,3], [0,2,1,3]], "desc": "标准图"},
        {"n": 2, "pre": [[1,0],[0,1]], "expected_one_of": [[]], "desc": "互为依赖（环）"},
        {"n": 3, "pre": [[1,0],[2,1],[0,2]], "expected_one_of": [[]], "desc": "三人成环"},
        {"n": 1, "pre": [], "expected_one_of": [[0]], "desc": "单门课"},
        {"n": 3, "pre": [], "expected_one_of": [[0,1,2], [2,1,0], [0,2,1]], "desc": "无依赖独立课"},
        {"n": 3, "pre": [[1,0],[1,2]], "expected_one_of": [[0,2,1], [2,0,1]], "desc": "多对一"},
        {"n": 2, "pre": [[0,1]], "expected_one_of": [[1,0]], "desc": "反向基础依赖"},
        {"n": 4, "pre": [[1,0],[2,0],[3,0]], "expected_one_of": [[0,1,2,3]], "desc": "一对多"},
        {"n": 3, "pre": [[1,0]], "expected_one_of": [[0,1,2], [2,0,1], [0,2,1]], "desc": "含孤立课程"}
    ]

    passed = 0
    print(f"{'ID':<4} | {'结果':<6} | {'描述'}")
    print("-" * 65)
    for i, tc in enumerate(test_cases):
        actual = func(tc["n"], tc["pre"])
        is_pass = any(actual == exp for exp in tc["expected_one_of"]) if tc["expected_one_of"] != [[]] else actual == []
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass: passed += 1
        print(f"{i+1:<4} | {status:<6} | {tc['desc']}")
    print("-" * 65)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_210(Solution210().findOrder)

ID   | 结果     | 描述
-----------------------------------------------------------------
1    | ✅ PASS | 基础依赖
2    | ✅ PASS | 标准图
3    | ✅ PASS | 互为依赖（环）
4    | ✅ PASS | 三人成环
5    | ✅ PASS | 单门课
6    | ✅ PASS | 无依赖独立课
7    | ✅ PASS | 多对一
8    | ✅ PASS | 反向基础依赖
9    | ✅ PASS | 一对多
10   | ✅ PASS | 含孤立课程
-----------------------------------------------------------------
测试总结: 通过 10/10


::: {.callout-important}
### 💡 思路讲解

1. **建立账本（数据结构准备）**：
   - **入度表 (In-degree)**：创建一个长度为 `n` 的数组，记录每门课还有几门“先修课”未完成。
   - **邻接表 (Adjacency List)**：记录每一门课是哪些后续课程的“通行证”。
   - **注意方向**：对于 `[ai, bi]`，`bi` 是先修课，`ai` 是后续课。所以 `indegree[ai] += 1`，且 `adj[bi]` 中加入 `ai`。

2. **寻找起点（初始化队列）**：
   - 遍历入度表，将所有 **入度为 0** 的课程（即没有任何先修课要求，可以直接上的课）放入队列 `q`。
   - 如果一开始所有课都有先修要求（且存在环），队列为空，搜索将直接结束。

3. **解锁流程（标准 BFS）**：
   - **弹出与记录**：从队列中弹出一门已“解锁”的课程 `curr`，将其加入结果列表 `res`。
   - **扩散与减压**：遍历 `adj[curr]`，找到所有依赖 `curr` 的后续课程 `next_course`。
   - **入度更新**：将 `next_course` 的入度减 1。
   - **判定新起点**：如果 `next_course` 的入度减到了 **0**，说明它的所有先修课都已修完，将其加入队列。

4. **结果判定（环检测）**：
   - 最终检查 `res` 中的课程数量是否等于 `numCourses`。
   - **相等**：说明所有课都顺利解锁，返回 `res`。
   - **不相等**：说明图中存在循环依赖（死锁），返回 `[]`。
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(V + E)$。$V$ 为课程数，$E$ 为先修关系数。每个节点和每条边仅被处理一次。
* **空间复杂度**：$O(V + E)$。用于存储邻接表和入度数组。
:::

## 📝 题目 1136：平行课程

::: {.callout-caution}
### 题目要求
**描述**：
给你 `n` 门课程，编号从 `1` 到 `n` 。同时给你一个数组 `relations` ，其中 `relations[i] = [prevCourse, nextCourse]` ，表示课程 `prevCourse` 必须在课程 `nextCourse` 之前修完。

在每一个学期中，你可以修读 **任意数量** 的课程，只要这些课程的所有先修课都已经在之前的学期里修完了。

**返回目标**：

- 返回修完所有课程所需的 **最少学期数** 。
- 如果无法修完所有课程（图中存在环），返回 `-1` 。

**输入输出示例**：

* **示例 1**：
    * **输入**：`n = 3, relations = [[1, 3], [2, 3]]`
    * **输出**：`2`
    * **说明**：在第一学期，修读课程 1 和 2 。在第二学期，修读课程 3 。

* **示例 2**：
    * **输入**：`n = 3, relations = [[1, 2], [2, 3], [3, 1]]`
    * **输出**：`-1`
    * **说明**：没有课程可以修读，因为它们形成了一个环。

* **示例 3**：
    * **输入**：`n = 2, relations = [[1, 2]]`
    * **输出**：`2`
    * **说明**：第一学期修 1，第二学期修 2。
:::

In [13]:
class Solution1136:
    def minimumSemesters(self, n: int, relations: List[List[int]]) -> int:
        indegree = [0] * (n + 1)  # 准备入度表，(编号 1~n，所以开 n+1 的空间)
        adj = [[] for _ in range(n + 1)]  # 准备邻接表
        for prev, nxt in relations:
            indegree[nxt] += 1  # 后续课的入度 +1
            adj[prev].append(nxt)  # 先修课指向后续课
        queue = deque([i for i in range(1, n + 1) if indegree[i] == 0])  # 寻找起点，注意不是 range(n)
        learned_count = 0
        semesters = 0
        while queue:
            semesters += 1
            for _ in range(len(queue)):  # 锁定当前学期能上的所有课
                curr = queue.popleft()
                learned_count += 1
                for nxt in adj[curr]:
                    indegree[nxt] -= 1  # 更新后续课的入度
                    if indegree[nxt] == 0:
                        queue.append(nxt)
        return semesters if learned_count == n else -1

In [14]:
#| code-fold: true
def test_1136(func):
    test_cases = [
        {"n": 3, "rel": [[1, 3], [2, 3]], "expected": 2, "desc": "标准并行依赖"},
        {"n": 3, "rel": [[1, 2], [2, 3], [3, 1]], "expected": -1, "desc": "简单三点环"},
        {"n": 2, "rel": [[1, 2]], "expected": 2, "desc": "两课线性依赖"},
        {"n": 4, "rel": [[1, 2], [3, 4]], "expected": 2, "desc": "两对独立关系"},
        {"n": 1, "rel": [], "expected": 1, "desc": "单门课程"},
        {"n": 5, "rel": [[1, 2], [2, 3], [3, 4], [4, 5]], "expected": 5, "desc": "深度线性链"},
        {"n": 3, "rel": [], "expected": 1, "desc": "全独立课程，一学期修完"},
        {"n": 4, "rel": [[1, 2], [1, 3], [2, 4], [3, 4]], "expected": 3, "desc": "钻石型依赖 (1->2,3->4)"},
        {"n": 3, "rel": [[1, 2], [2, 1]], "expected": -1, "desc": "两点互等死循环"},
        {"n": 6, "rel": [[1, 2], [1, 3], [2, 4], [3, 4], [4, 5], [4, 6]], "expected": 4, "desc": "复杂混合图"}
    ]
    passed = 0
    print(f"{'ID':<4} | {'结果':<6} | {'预期':<4} | {'实际':<4} | {'描述'}")
    print("-" * 65)
    for i, tc in enumerate(test_cases):
        actual = func(tc["n"], tc["rel"])
        is_pass = actual == tc["expected"]
        status = "✅ PASS" if is_pass else "❌ FAIL"
        if is_pass: passed += 1
        print(f"{i+1:<4} | {status:<6} | {tc['expected']:<4} | {actual:<4} | {tc['desc']}")
    print("-" * 65)
    print(f"测试总结: 通过 {passed}/{len(test_cases)}")

# 运行测试
test_1136(Solution1136().minimumSemesters)

ID   | 结果     | 预期   | 实际   | 描述
-----------------------------------------------------------------
1    | ✅ PASS | 2    | 2    | 标准并行依赖
2    | ✅ PASS | -1   | -1   | 简单三点环
3    | ✅ PASS | 2    | 2    | 两课线性依赖
4    | ✅ PASS | 2    | 2    | 两对独立关系
5    | ✅ PASS | 1    | 1    | 单门课程
6    | ✅ PASS | 5    | 5    | 深度线性链
7    | ✅ PASS | 1    | 1    | 全独立课程，一学期修完
8    | ✅ PASS | 3    | 3    | 钻石型依赖 (1->2,3->4)
9    | ✅ PASS | -1   | -1   | 两点互等死循环
10   | ✅ PASS | 4    | 4    | 复杂混合图
-----------------------------------------------------------------
测试总结: 通过 10/10


::: {.callout-important}
### 💡 思路讲解

1. **建模与编号对齐**：
   - 课程编号为 $1 \sim n$，为了方便，我们创建长度为 $n+1$ 的入度数组和邻接表，直接无视下标 `0`。
   - 遍历 `relations`：`prev` 指向 `next`，`next` 的入度增加。

2. **学期划分（层序 BFS）**：
   - 使用标准 BFS 队列。初始时，将所有入度为 0 的课程入队。
   - **关键操作**：使用 `for _ in range(len(q))` 锁定当前学期能上的所有课。
   - 每处理完这样的一层，学期数 `semesters` 加 1。

3. **环检测与结果**：
   - 记录修读过的课程总数 `count`。
   - 如果 `count == n`，返回 `semesters`；否则说明有课没修完（有环），返回 `-1`。
:::

::: {.callout-note}
### 💡 复杂度分析
* **时间复杂度**：$O(V + E)$。每门课和每个关系只处理一次。
* **空间复杂度**：$O(V + E)$。存储图结构所需的空间。
:::